### Transformer from scratch using Pytorch

https://medium.com/@bavalpreetsinghh/transformer-from-scratch-using-pytorch-28a5d1b2e033

In [1]:
import torch
import torch.nn as nn
import math

d_model = 512
N = 6
h = 8
dropout = 0.1
d_ff = 2048
src_vocab_size = 50000
tgt_vocab_size = 50000
src_seq_len = 128
tgt_seq_len = 128

In [2]:
class InputEmbeddings(nn.Module):

    def __init__(self, d_model: int, vocab_size: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        # (batch, seq_len) --> (batch, seq_len, d_model)
        # Multiply by sqrt(d_model) to scale the embeddings according to the paper
        return self.embedding(x) * math.sqrt(self.d_model)

In [3]:
seq_len=6
d_model= 6
pe = torch.zeros(seq_len, d_model)

In [4]:
position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1) # (seq_len, 1)

In [5]:
position

tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.],
        [5.]])

In [6]:
torch.arange(0, 5, dtype=torch.float).unsqueeze(0)

tensor([[0., 1., 2., 3., 4.]])

In [7]:
torch.arange(0, 5, dtype=torch.float).unsqueeze(1)

tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.]])

In [8]:
pe

tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])

In [9]:
torch.arange(0, d_model, 2).float()

tensor([0., 2., 4.])

In [10]:
pe[:, 0::2] = 2.0

In [11]:
pe

tensor([[2., 0., 2., 0., 2., 0.],
        [2., 0., 2., 0., 2., 0.],
        [2., 0., 2., 0., 2., 0.],
        [2., 0., 2., 0., 2., 0.],
        [2., 0., 2., 0., 2., 0.],
        [2., 0., 2., 0., 2., 0.]])

In [12]:
two_i = torch.arange(0, d_model, 2).float()
two_i

tensor([0., 2., 4.])

In [13]:
# div_term = -((2i/d) * math.log(10000))
# div_term = exp(log(1/10000**2i/d_model)) = exp(-2i/d_model* log(10000)
div_term = torch.exp(-two_i / d_model * math.log(10000.0)) 

In [14]:
div_term

tensor([1.0000, 0.0464, 0.0022])

In [15]:
position

tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.],
        [5.]])

In [16]:
position * div_term

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 4.6416e-02, 2.1544e-03],
        [2.0000e+00, 9.2832e-02, 4.3089e-03],
        [3.0000e+00, 1.3925e-01, 6.4633e-03],
        [4.0000e+00, 1.8566e-01, 8.6177e-03],
        [5.0000e+00, 2.3208e-01, 1.0772e-02]])

In [17]:
torch.sin(position * div_term)

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.8415,  0.0464,  0.0022],
        [ 0.9093,  0.0927,  0.0043],
        [ 0.1411,  0.1388,  0.0065],
        [-0.7568,  0.1846,  0.0086],
        [-0.9589,  0.2300,  0.0108]])

In [18]:
torch.cos(position * div_term)

tensor([[ 1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.9989,  1.0000],
        [-0.4161,  0.9957,  1.0000],
        [-0.9900,  0.9903,  1.0000],
        [-0.6536,  0.9828,  1.0000],
        [ 0.2837,  0.9732,  0.9999]])

In [19]:
pe

tensor([[2., 0., 2., 0., 2., 0.],
        [2., 0., 2., 0., 2., 0.],
        [2., 0., 2., 0., 2., 0.],
        [2., 0., 2., 0., 2., 0.],
        [2., 0., 2., 0., 2., 0.],
        [2., 0., 2., 0., 2., 0.]])

In [20]:
pe = torch.zeros(seq_len, d_model)
pe

tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])

In [21]:
pe[:,0::2] = torch.sin(position * div_term)
pe

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.8415,  0.0000,  0.0464,  0.0000,  0.0022,  0.0000],
        [ 0.9093,  0.0000,  0.0927,  0.0000,  0.0043,  0.0000],
        [ 0.1411,  0.0000,  0.1388,  0.0000,  0.0065,  0.0000],
        [-0.7568,  0.0000,  0.1846,  0.0000,  0.0086,  0.0000],
        [-0.9589,  0.0000,  0.2300,  0.0000,  0.0108,  0.0000]])

In [22]:
pe[:, 1::2] = torch.cos(position * div_term)
pe

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000],
        [-0.7568, -0.6536,  0.1846,  0.9828,  0.0086,  1.0000],
        [-0.9589,  0.2837,  0.2300,  0.9732,  0.0108,  0.9999]])

In [23]:
pe = pe.unsqueeze(0)

In [24]:
pe

tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
         [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
         [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000],
         [-0.7568, -0.6536,  0.1846,  0.9828,  0.0086,  1.0000],
         [-0.9589,  0.2837,  0.2300,  0.9732,  0.0108,  0.9999]]])

In [25]:
pe.shape[1]

6

In [26]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        # Create a matrix of shape (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)
        # Create a vector of shape (seq_len)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1) # (seq_len, 1)
        # Create a vector of shape (d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # (d_model / 2)
        # Apply sine to even indices
        pe[:, 0::2] = torch.sin(position * div_term) # sin(position * (10000 ** (2i / d_model))
        # Apply cosine to odd indices
        pe[:, 1::2] = torch.cos(position * div_term) # cos(position * (10000 ** (2i / d_model))
        # Add a batch dimension to the positional encoding
        pe = pe.unsqueeze(0) # (1, seq_len, d_model)
        # Register the positional encoding as a buffer
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False) # (batch, seq_len, d_model)
        return self.dropout(x)

In [27]:
pos_embedding = PositionalEncoding(6, 6, 0.50)
pos_embedding 

PositionalEncoding(
  (dropout): Dropout(p=0.5, inplace=False)
)

In [28]:
# Create dummy input with shape (batch_size, seq_len, d_model)
input_data = torch.randn(32, 10, 512)  # 32 sentences, each with 10 tokens, each token represented by a 512-dimensional embedding

In [29]:
input_data = torch.randn(2, 6, 6)
pos_emd = pos_embedding(input_data)
pos_emd

tensor([[[-2.1064,  0.0000,  0.0000,  0.0000,  0.0000,  1.1668],
         [ 0.0000, -0.1750, -0.0000,  0.0000,  0.0000,  2.4535],
         [ 2.1927, -0.3451,  0.0000, -0.0000,  0.0000,  0.0000],
         [-0.0000, -0.0000, -0.0000,  0.0000,  0.6119,  0.0000],
         [ 0.0000,  2.3135,  0.7860,  0.0000,  0.0000,  5.1914],
         [-4.1786,  0.0000, -0.0000,  4.3897,  1.0379, -0.0000]],

        [[-1.5701,  4.2349,  1.9697,  0.0000, -0.0000,  0.0000],
         [ 0.4361, -2.2855, -0.0000,  0.0000, -1.4957,  0.0000],
         [ 0.0000,  0.2922,  0.0000,  3.2878, -0.0000,  0.0000],
         [ 2.9402,  2.5173,  0.0000,  0.0000,  0.1921,  0.0000],
         [ 0.0187,  0.0000, -1.1596,  2.2259,  0.0000, -0.3413],
         [-0.0000,  0.2192,  0.0000,  3.5579, -0.0000,  0.0000]]])

In [30]:
input_data.shape

torch.Size([2, 6, 6])

In [31]:
pe.shape

torch.Size([1, 6, 6])

In [32]:
input_data

tensor([[[-1.0532e+00, -7.3594e-01,  2.0904e+00,  2.4823e-01,  1.9681e+00,
          -4.1660e-01],
         [ 1.6987e+00, -6.2781e-01, -1.2391e+00, -9.6174e-01,  4.2212e-01,
           2.2674e-01],
         [ 1.8704e-01,  2.4361e-01,  7.7513e-03, -1.9924e+00,  1.4054e+00,
          -3.5509e-03],
         [-3.9685e-01, -6.1198e-01, -1.5534e+00, -1.1916e-03,  2.9948e-01,
           1.9821e-02],
         [ 1.2574e+00,  1.8104e+00,  2.0842e-01, -5.6268e-02,  1.7534e-01,
           1.5957e+00],
         [-1.1304e+00,  2.1331e-01, -2.5653e-01,  1.2217e+00,  5.0818e-01,
          -2.5748e+00]],

        [[-7.8503e-01,  1.1175e+00,  9.8483e-01, -4.1291e-01, -9.0878e-01,
           1.2842e+00],
         [-6.2344e-01, -1.6830e+00, -1.5798e+00, -4.9780e-02, -7.5002e-01,
           6.8196e-01],
         [ 5.4800e-01,  5.6222e-01,  5.8807e-01,  6.4821e-01, -3.7502e-01,
          -4.2246e-01],
         [ 1.3290e+00,  2.2486e+00,  6.1567e-02,  5.2593e-01,  8.9593e-02,
           7.0460e-01],
        

In [33]:
input_data.mean(dim = -1, keepdim = True)

tensor([[[ 0.3502],
         [-0.0802],
         [-0.0254],
         [-0.3740],
         [ 0.8318],
         [-0.3364]],

        [[ 0.2133],
         [-0.6674],
         [ 0.2582],
         [ 0.8265],
         [ 0.1724],
         [-0.1875]]])

In [34]:
input_data.mean(dim = 1, keepdim = True)

tensor([[[ 0.0938,  0.0486, -0.1237, -0.2570,  0.7964, -0.1921]],

        [[ 0.0475,  0.4699, -0.0731,  0.2746, -0.3136,  0.2104]]])

In [35]:
mean = input_data.mean(dim = -1, keepdim = True)
mean

tensor([[[ 0.3502],
         [-0.0802],
         [-0.0254],
         [-0.3740],
         [ 0.8318],
         [-0.3364]],

        [[ 0.2133],
         [-0.6674],
         [ 0.2582],
         [ 0.8265],
         [ 0.1724],
         [-0.1875]]])

In [36]:
std = input_data.std(dim = -1, keepdim = True)
std

tensor([[[1.3707],
         [1.0891],
         [1.0987],
         [0.6627],
         [0.8162],
         [1.3484]],

        [[1.0205],
         [0.9028],
         [0.5102],
         [0.8377],
         [0.9691],
         [0.7839]]])

In [37]:
class LayerNormalization(nn.Module):

    def __init__(self, features: int, eps:float=10**-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features)) # alpha is a learnable parameter
        self.bias = nn.Parameter(torch.zeros(features)) # bias is a learnable parameter

    def forward(self, x):
        # x: (batch, seq_len, hidden_size)
         # Keep the dimension for broadcasting
        mean = x.mean(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # Keep the dimension for broadcasting
        std = x.std(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # eps is to prevent dividing by zero or when std is very small
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

In [38]:
class FeedForwardBlock(nn.Module):

    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff) # w1 and b1
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model) # w2 and b2

    def forward(self, x):
        # (batch, seq_len, d_model) --> (batch, seq_len, d_ff) --> (batch, seq_len, d_model)
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

In [39]:
query = torch.randn(2,3,8)
key = torch.randn(2,3,8)
value = torch.randn(2,3,8)

In [40]:
h = 2
d_k = 4
query = query.view(query.shape[0], query.shape[1], h, d_k).transpose(1, 2)
key = key.view(key.shape[0], key.shape[1], h, d_k).transpose(1, 2)
value = value.view(value.shape[0], value.shape[1], h, d_k).transpose(1, 2)

In [41]:
query.shape

torch.Size([2, 2, 3, 4])

In [42]:
query

tensor([[[[ 0.2824, -0.0750, -2.2568, -0.4100],
          [-0.0784,  0.2555, -1.2746,  0.6654],
          [ 0.3549, -0.3465,  0.2911,  1.6185]],

         [[ 0.0603,  0.0580,  0.0362, -1.7849],
          [-2.1000,  0.8092,  0.6741, -2.3694],
          [-1.4095,  1.1164,  0.2286,  0.0402]]],


        [[[ 1.0216,  0.3559,  0.3654, -1.0391],
          [ 0.1635,  1.4950,  0.4933,  0.3320],
          [-0.1107,  0.1624, -0.7899, -0.1558]],

         [[ 0.6211, -0.4074,  0.5780, -0.2910],
          [ 0.9018,  0.3314, -0.1550,  2.1025],
          [-0.0749,  1.4327,  0.1096, -0.3091]]]])

In [43]:
key.transpose(-2, -1).shape

torch.Size([2, 2, 4, 3])

In [44]:
key.transpose(2, 3)

tensor([[[[-0.7616,  0.1097,  0.8037],
          [-2.6329,  2.0541, -0.1552],
          [-2.7019, -1.4497, -0.2273],
          [ 0.3256, -0.6643, -1.0227]],

         [[ 0.0075,  0.2830,  0.2401],
          [-0.4311, -0.0876, -0.3979],
          [-0.4530, -1.2323, -1.8301],
          [-0.0155, -1.3821,  0.6286]]],


        [[[ 0.7738, -3.8168,  1.0149],
          [ 0.1713,  2.5963,  1.2830],
          [-1.2890,  0.5623, -0.9894],
          [ 0.6480, -0.0668, -0.8087]],

         [[-0.1154,  2.3446, -1.1461],
          [ 0.1380, -0.2777, -0.3379],
          [-0.3486,  0.1032,  0.2823],
          [ 0.5279,  1.0145, -1.1672]]]])

In [45]:
key.transpose(3, 2)

tensor([[[[-0.7616,  0.1097,  0.8037],
          [-2.6329,  2.0541, -0.1552],
          [-2.7019, -1.4497, -0.2273],
          [ 0.3256, -0.6643, -1.0227]],

         [[ 0.0075,  0.2830,  0.2401],
          [-0.4311, -0.0876, -0.3979],
          [-0.4530, -1.2323, -1.8301],
          [-0.0155, -1.3821,  0.6286]]],


        [[[ 0.7738, -3.8168,  1.0149],
          [ 0.1713,  2.5963,  1.2830],
          [-1.2890,  0.5623, -0.9894],
          [ 0.6480, -0.0668, -0.8087]],

         [[-0.1154,  2.3446, -1.1461],
          [ 0.1380, -0.2777, -0.3379],
          [-0.3486,  0.1032,  0.2823],
          [ 0.5279,  1.0145, -1.1672]]]])

In [46]:
key.transpose(2, 3) == key.transpose(-2, -1)

tensor([[[[True, True, True],
          [True, True, True],
          [True, True, True],
          [True, True, True]],

         [[True, True, True],
          [True, True, True],
          [True, True, True],
          [True, True, True]]],


        [[[True, True, True],
          [True, True, True],
          [True, True, True],
          [True, True, True]],

         [[True, True, True],
          [True, True, True],
          [True, True, True],
          [True, True, True]]]])

In [47]:
attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

In [48]:
attention_scores.shape

torch.Size([2, 2, 3, 3])

In [49]:
attention_scores

tensor([[[[ 2.9733e+00,  1.7105e+00,  5.8543e-01],
          [ 1.5237e+00,  9.6102e-01, -2.4673e-01],
          [ 1.9117e-01, -1.0850e+00, -6.9119e-01]],

         [[-6.6019e-03,  1.2172e+00, -5.9838e-01],
          [-3.1657e-01,  8.8950e-01, -1.7745e+00],
          [-2.9803e-01, -4.1696e-01, -5.8781e-01]]],


        [[[-1.4645e-01, -1.3502e+00,  9.8614e-01],
          [-1.9082e-02,  1.7563e+00,  6.6373e-01],
          [ 4.2964e-01,  2.0519e-01,  5.0172e-01]],

         [[-2.4151e-01,  6.6691e-01, -3.5697e-02],
          [ 5.5282e-01,  2.0697e+00, -1.8216e+00],
          [ 2.5220e-03, -4.3783e-01, -3.2883e-03]]]])

In [50]:
attention_scores.shape

torch.Size([2, 2, 3, 3])

In [51]:
attention_scores = attention_scores.softmax(dim=-1) # (batch, h, seq_len, seq_len) # Apply softmax
attention_scores.shape

torch.Size([2, 2, 3, 3])

In [52]:
value.shape

torch.Size([2, 2, 3, 4])

In [53]:
x = attention_scores @ value
x.shape

torch.Size([2, 2, 3, 4])

In [54]:
x.transpose(1, 2).shape

torch.Size([2, 3, 2, 4])

In [55]:
x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, h * d_k)
x.shape

torch.Size([2, 3, 8])

In [56]:
class MultiHeadAttentionBlock(nn.Module):

    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model # Embedding vector size
        self.h = h # Number of heads
        # Make sure d_model is divisible by h
        assert d_model % h == 0, "d_model is not divisible by h"

        self.d_k = d_model // h # Dimension of vector seen by each head
        self.w_q = nn.Linear(d_model, d_model, bias=False) # Wq
        self.w_k = nn.Linear(d_model, d_model, bias=False) # Wk
        self.w_v = nn.Linear(d_model, d_model, bias=False) # Wv
        self.w_o = nn.Linear(d_model, d_model, bias=False) # Wo
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        # Just apply the formula from the paper
        # (batch, h, seq_len, d_k) --> (batch, h, seq_len, seq_len)
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            # Write a very low value (indicating -inf) to the positions where mask == 0
            attention_scores.masked_fill_(mask == 0, -1e9)
        attention_scores = attention_scores.softmax(dim=-1) # (batch, h, seq_len, seq_len) # Apply softmax
        if dropout is not None:
            attention_scores = dropout(attention_scores)
        # (batch, h, seq_len, seq_len) --> (batch, h, seq_len, d_k)
        # return attention scores which can be used for visualization
        return (attention_scores @ value), attention_scores

    def forward(self, q, k, v, mask):
        query = self.w_q(q) # (batch, seq_len, d_model) --> (batch, seq_len, d_model)
        key = self.w_k(k) # (batch, seq_len, d_model) --> (batch, seq_len, d_model)
        value = self.w_v(v) # (batch, seq_len, d_model) --> (batch, seq_len, d_model)

        # (batch, seq_len, d_model) --> (batch, seq_len, h, d_k) --> (batch, h, seq_len, d_k)
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        # Calculate attention
        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)
        
        # Combine all the heads together
        # (batch, h, seq_len, d_k) --> (batch, seq_len, h, d_k) --> (batch, seq_len, d_model)
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)

        # Multiply by Wo
        # (batch, seq_len, d_model) --> (batch, seq_len, d_model)  
        return self.w_o(x)

In [57]:
# Example input sequences (batch_size=2, seq_len=5)
input_sequences = [
    [1, 2, 3, 0, 0],  # Sequence 1, padded with 0s
    [4, 5, 6, 7, 8]   # Sequence 2, no padding
]
# Convert to tensor
input_tensor = torch.tensor(input_sequences)
# Create a padding mask where 1 indicates a valid position and 0 indicates padding

In [58]:
mask = (input_tensor != 0)
print(mask)
print(mask.shape)

tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True,  True]])
torch.Size([2, 5])


In [59]:
mask.unsqueeze(1).shape

torch.Size([2, 1, 5])

In [60]:
mask.unsqueeze(1).unsqueeze(2).shape

torch.Size([2, 1, 1, 5])

In [61]:
mask.unsqueeze(1).unsqueeze(2)

tensor([[[[ True,  True,  True, False, False]]],


        [[[ True,  True,  True,  True,  True]]]])

In [62]:
seq_len = 5
torch.ones(seq_len, seq_len)

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

In [63]:
torch.triu(torch.ones(seq_len, seq_len), diagonal=1)

tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])

In [64]:
attn_score = torch.ones(5,5)

In [65]:
attn_score

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

In [66]:
import torch
# Create a look-ahead mask
seq_len = 5
look_ahead_mask = torch.triu(torch.ones((seq_len, seq_len)), diagonal=1).bool()
print("Look-Ahead Mask:")
print(look_ahead_mask)

Look-Ahead Mask:
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])


In [67]:
attn_score.masked_fill_(look_ahead_mask, -1e9)
attn_score

tensor([[ 1.0000e+00, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [ 1.0000e+00,  1.0000e+00, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [ 1.0000e+00,  1.0000e+00,  1.0000e+00, -1.0000e+09, -1.0000e+09],
        [ 1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00, -1.0000e+09],
        [ 1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00]])

In [68]:
attn_score.masked_fill_(look_ahead_mask == 0, -1e9)
attn_score

tensor([[-1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [-1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [-1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [-1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [-1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09]])

In [69]:
class ResidualConnection(nn.Module):
    
        def __init__(self, features: int, dropout: float) -> None:
            super().__init__()
            self.dropout = nn.Dropout(dropout)
            self.norm = LayerNormalization(features)
    
        def forward(self, x, sublayer):
            return x + self.dropout(sublayer(self.norm(x)))

In [70]:
class EncoderBlock(nn.Module):

    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(2)])

    def forward(self, x, src_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, src_mask))
        x = self.residual_connections[1](x, self.feed_forward_block)
        return x

In [71]:
input_data = torch.randn(2,128,8)
input_data

tensor([[[ 0.5261, -0.1952,  0.2990,  ..., -3.3856, -1.0592, -0.2043],
         [ 0.4410,  1.1951,  1.1355,  ...,  1.0401, -0.5401, -0.8870],
         [ 1.4823, -2.0673,  0.5101,  ..., -0.2935, -1.1986,  1.7744],
         ...,
         [ 1.6214,  0.1172, -2.0246,  ..., -0.5141, -0.2214, -0.6910],
         [ 0.4125,  1.3205,  1.5449,  ..., -1.6260,  1.7191,  0.4729],
         [-0.3104, -2.6669, -0.0217,  ..., -0.9436, -0.1083, -0.6026]],

        [[ 0.8670,  0.4904,  1.8629,  ..., -0.8429, -0.9042,  0.5202],
         [-1.2285,  0.5007, -0.6760,  ..., -0.3534,  0.4718, -0.8590],
         [ 0.8731, -0.1983,  0.0726,  ..., -1.3501,  1.6027, -0.8572],
         ...,
         [ 0.1882,  0.0773,  0.5210,  ...,  0.2662,  0.3871,  1.0841],
         [ 0.9343,  0.3758,  0.2667,  ..., -1.1454, -0.2741,  0.0064],
         [-0.0152, -1.2272,  0.1715,  ...,  1.8346,  0.7201, -1.1931]]])

In [72]:
d_model = 512
src_embed = InputEmbeddings(d_model, src_vocab_size)
tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)

In [73]:
src_embed

InputEmbeddings(
  (embedding): Embedding(50000, 512)
)

In [74]:
src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)

In [75]:
encoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
encoder_block = EncoderBlock(d_model, encoder_self_attention_block, feed_forward_block, dropout)

In [76]:
input_data = torch.randint(0,49999,(2,128))
print(input_data)
print(input_data.shape)

tensor([[20843, 17806,  9828,  4652, 23668, 12299, 41321, 18806,  9409, 22586,
         32894, 32433, 37696, 30761, 43625, 21478,  4665, 11713, 43978, 11625,
         28335,  4463, 24365, 34943, 40305, 47438, 19527, 26861, 46634, 13368,
         42809, 40504, 49621,  8674, 29354,  3677, 41787, 30104, 49180, 19910,
         32918, 41050, 22284, 12007, 41286,  9673,  8856, 25236, 30046, 29233,
          8765,  7765, 44169, 47722, 41290, 43764, 41200,  7212, 26420,   702,
         37108, 43541, 14430, 10227, 39715, 14772, 31473, 12136, 29225, 18278,
         18828, 40503, 39643, 11553, 10914, 22035, 41271,  9456,  9464, 11221,
         48265, 41121, 19298, 32043, 29592, 26367,  5041, 43403, 38098,  5581,
         23417, 14471, 30364, 45586, 42321, 35576, 45832, 14626, 46642, 23556,
         19501, 28134,  1733,  9086, 33829, 49288,  3492, 34137, 34765,  7517,
         17519,  8989, 48642,  6484, 45912, 43499, 48384, 35061, 22451,  5155,
         12099, 24533, 15137, 44680,   834,  9123, 4

In [77]:
x = src_embed(input_data)
x.shape

torch.Size([2, 128, 512])

In [78]:
x = src_pos(x)
x.shape

torch.Size([2, 128, 512])

In [79]:
x = encoder_block(x, src_mask= None)
x.shape

torch.Size([2, 128, 512])

In [80]:
class LayerNormalization(nn.Module):

    def __init__(self, features: int, eps:float=10**-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features)) # alpha is a learnable parameter
        self.bias = nn.Parameter(torch.zeros(features)) # bias is a learnable parameter

    def forward(self, x):
        # x: (batch, seq_len, hidden_size)
         # Keep the dimension for broadcasting
        mean = x.mean(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # Keep the dimension for broadcasting
        std = x.std(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # eps is to prevent dividing by zero or when std is very small
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

class FeedForwardBlock(nn.Module):

    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff) # w1 and b1
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model) # w2 and b2

    def forward(self, x):
        # (batch, seq_len, d_model) --> (batch, seq_len, d_ff) --> (batch, seq_len, d_model)
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

class InputEmbeddings(nn.Module):

    def __init__(self, d_model: int, vocab_size: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        # (batch, seq_len) --> (batch, seq_len, d_model)
        # Multiply by sqrt(d_model) to scale the embeddings according to the paper
        return self.embedding(x) * math.sqrt(self.d_model)
    
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        # Create a matrix of shape (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)
        # Create a vector of shape (seq_len)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1) # (seq_len, 1)
        # Create a vector of shape (d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # (d_model / 2)
        # Apply sine to even indices
        pe[:, 0::2] = torch.sin(position * div_term) # sin(position * (10000 ** (2i / d_model))
        # Apply cosine to odd indices
        pe[:, 1::2] = torch.cos(position * div_term) # cos(position * (10000 ** (2i / d_model))
        # Add a batch dimension to the positional encoding
        pe = pe.unsqueeze(0) # (1, seq_len, d_model)
        # Register the positional encoding as a buffer
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False) # (batch, seq_len, d_model)
        return self.dropout(x)

class ResidualConnection(nn.Module):
    
        def __init__(self, features: int, dropout: float) -> None:
            super().__init__()
            self.dropout = nn.Dropout(dropout)
            self.norm = LayerNormalization(features)
    
        def forward(self, x, sublayer):
            return x + self.dropout(sublayer(self.norm(x)))

class MultiHeadAttentionBlock(nn.Module):

    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model # Embedding vector size
        self.h = h # Number of heads
        # Make sure d_model is divisible by h
        assert d_model % h == 0, "d_model is not divisible by h"

        self.d_k = d_model // h # Dimension of vector seen by each head
        self.w_q = nn.Linear(d_model, d_model, bias=False) # Wq
        self.w_k = nn.Linear(d_model, d_model, bias=False) # Wk
        self.w_v = nn.Linear(d_model, d_model, bias=False) # Wv
        self.w_o = nn.Linear(d_model, d_model, bias=False) # Wo
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        # Just apply the formula from the paper
        # (batch, h, seq_len, d_k) --> (batch, h, seq_len, seq_len)
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            # Write a very low value (indicating -inf) to the positions where mask == 0
            attention_scores.masked_fill_(mask == 0, -1e9)
        attention_scores = attention_scores.softmax(dim=-1) # (batch, h, seq_len, seq_len) # Apply softmax
        if dropout is not None:
            attention_scores = dropout(attention_scores)
        # (batch, h, seq_len, seq_len) --> (batch, h, seq_len, d_k)
        # return attention scores which can be used for visualization
        return (attention_scores @ value), attention_scores

    def forward(self, q, k, v, mask):
        query = self.w_q(q) # (batch, seq_len, d_model) --> (batch, seq_len, d_model)
        key = self.w_k(k) # (batch, seq_len, d_model) --> (batch, seq_len, d_model)
        value = self.w_v(v) # (batch, seq_len, d_model) --> (batch, seq_len, d_model)

        # (batch, seq_len, d_model) --> (batch, seq_len, h, d_k) --> (batch, h, seq_len, d_k)
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        # Calculate attention
        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)
        
        # Combine all the heads together
        # (batch, h, seq_len, d_k) --> (batch, seq_len, h, d_k) --> (batch, seq_len, d_model)
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)

        # Multiply by Wo
        # (batch, seq_len, d_model) --> (batch, seq_len, d_model)  
        return self.w_o(x)

class EncoderBlock(nn.Module):

    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(2)])

    def forward(self, x, src_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, src_mask))
        x = self.residual_connections[1](x, self.feed_forward_block)
        return x
    
class Encoder(nn.Module):

    def __init__(self, features: int, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderBlock(nn.Module):

    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock, cross_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(3)])

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, tgt_mask))
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connections[2](x, self.feed_forward_block)
        return x
    
class Decoder(nn.Module):

    def __init__(self, features: int, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

class ProjectionLayer(nn.Module):

    def __init__(self, d_model, vocab_size) -> None:
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x) -> None:
        # (batch, seq_len, d_model) --> (batch, seq_len, vocab_size)
        return self.proj(x)
    
class Transformer(nn.Module):

    def __init__(self, encoder: Encoder, decoder: Decoder, src_embed: InputEmbeddings, tgt_embed: InputEmbeddings, src_pos: PositionalEncoding, tgt_pos: PositionalEncoding, projection_layer: ProjectionLayer) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer

    def encode(self, src, src_mask):
        # (batch, seq_len, d_model)
        src = self.src_embed(src)
        src = self.src_pos(src)
        return self.encoder(src, src_mask)
    
    def decode(self, encoder_output: torch.Tensor, src_mask: torch.Tensor, tgt: torch.Tensor, tgt_mask: torch.Tensor):
        # (batch, seq_len, d_model)
        tgt = self.tgt_embed(tgt)
        tgt = self.tgt_pos(tgt)
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)
    
    def project(self, x):
        # (batch, seq_len, vocab_size)
        return self.projection_layer(x)
    
def build_transformer(src_vocab_size: int, tgt_vocab_size: int, src_seq_len: int, tgt_seq_len: int, d_model: int=512, N: int=6, h: int=8, dropout: float=0.1, d_ff: int=2048) -> Transformer:
    # Create the embedding layers
    src_embed = InputEmbeddings(d_model, src_vocab_size)
    tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)

    # Create the positional encoding layers
    src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
    tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)
    
    # Create the encoder blocks
    encoder_blocks = []
    for _ in range(N):
        encoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
        encoder_block = EncoderBlock(d_model, encoder_self_attention_block, feed_forward_block, dropout)
        encoder_blocks.append(encoder_block)

    # Create the decoder blocks
    decoder_blocks = []
    for _ in range(N):
        decoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
        decoder_cross_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
        decoder_block = DecoderBlock(d_model, decoder_self_attention_block, decoder_cross_attention_block, feed_forward_block, dropout)
        decoder_blocks.append(decoder_block)
    
    # Create the encoder and decoder
    encoder = Encoder(d_model, nn.ModuleList(encoder_blocks))
    decoder = Decoder(d_model, nn.ModuleList(decoder_blocks))
    
    # Create the projection layer
    projection_layer = ProjectionLayer(d_model, tgt_vocab_size)
    
    # Create the transformer
    transformer = Transformer(encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, projection_layer)
    
    # Initialize the parameters
    for p in transformer.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    
    return transformer

In [81]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [82]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

In [83]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

### In Summary

In [1]:
import torch
import torch.nn as nn
import math

class InputEmbeddings(nn.Module):

    def __init__(self, d_model: int, vocab_size: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model) 

    def forward(self, x):
        #print(x.shape)
        return self.embedding(x) * math.sqrt(self.d_model)
    
    

class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout) 

        pe = torch.zeros(self.seq_len, self.d_model)
        position = torch.arange(0, self.seq_len, dtype=torch.float).unsqueeze(1) #(seq_len,1)
        div_term = torch.exp(torch.arange(0, self.d_model, 2).float() / -self.d_model * math.log(10000.0))
        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0) # (1, seq_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x)


class LayerNormalization(nn.Module):

    def __init__(self, features: int, eps: float = 10**-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.zeros(features))
        self.bias = nn.Parameter(torch.zeros(features))

    def forward(self, x):
        mean = x.mean(dim = -1, keepdim=True)
        std = x.std(dim = -1, keepdim=True)
        
        return self.alpha * (x - mean) / (std + self.eps) + self.bias


class FeedForwardBlock(nn.Module):
    
    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))


class MultiHeadAttentionBlock(nn.Module):
    
    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0, "d_model is not divisible by h"
        
        self.d_k = d_model // h
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model) 
        self.dropout = nn.Dropout(dropout)


    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):

        d_k = query.shape[-1]
        attention_scores = (query @ key.transpose(-2,-1)) / math.sqrt(d_k)
        if mask is not None:
            attention_scores.masked_fill_(mask == 0, -1e9)

        attention_scores.softmax(dim = -1)
        if dropout is not None:
            attention_scores = dropout(attention_scores)

        #print(f"attn_scores.shape : {attention_scores.shape}")
        #print(f"value.shape: {value.shape}")
        return (attention_scores @ value), attention_scores    

    def forward(self, q, k, v, mask):
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1,2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1,2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1,2)

        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

        # (batch, h, seq_len, d_k) -> (batch, seq_len, h, d_k) -> (batch, seq_len, h*d_k)
        x = x.transpose(1, 2).contiguous().view(x.shape[0],-1, self.h * self.d_k)

        # (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        return self.w_o(x)



class ResidualConnection(nn.Module):

    def __init__(self, features: int, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNormalization(features)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))


class EncoderBlock(nn.Module):

    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock, feedforward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feedforward_block = feedforward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(2)])

    def forward(self, x, src_mask):
        # x = x + self.self_attention_block(x, x, x, src_mask)
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, src_mask))
        x = self.residual_connections[1](x, self.feedforward_block)
        return x



class Encoder(nn.Module):

    def __init__(self, features: int, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self,x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)



class DecoderBlock(nn.Module):

    def __init__(self, features: int, self_attention_block: MultiHeadAttentionBlock, cross_attention_block: MultiHeadAttentionBlock, feedforward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feedforward_block = feedforward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(3)])


    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, tgt_mask))
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connections[2](x, self.feedforward_block)

        return x


class Decoder(nn.Module):

    def __init__(self, features: int, layers:nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        
        return self.norm(x)



class ProjectionLayer(nn.Module):

    def __init__(self, d_model: int, vocab_size: int) -> None:
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        return torch.log_softmax(self.proj(x), dim = -1) 
    
    

class Transformer(nn.Module):

    def __init__(self, encoder: Encoder, decoder: Decoder, src_embed: InputEmbeddings, tgt_embed: InputEmbeddings, src_pos: PositionalEncoding, tgt_pos: PositionalEncoding, projection_layer: ProjectionLayer) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer


    def encode(self, src, mask):
        src = self.src_embed(src)
        src = self.src_pos(src)
        return self.encoder(src, mask)

    def decode(self, encoder_output, src_mask, tgt, tgt_mask):
        #print(f"decode function: tgt.shape: {tgt.shape}")
        tgt = self.tgt_embed(tgt)
        tgt = self.tgt_pos(tgt) 
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)

    def project(self, x):
        return self.projection_layer(x)


def build_transformer(src_vocab_size: int, tgt_vocab_size: int, src_seq_len: int, tgt_seq_len: int, d_model: int = 512, h: int = 8, N: int = 6, dropout: float = 0.1, d_ff: int = 2048) -> Transformer:
    src_embed = InputEmbeddings(d_model, src_vocab_size)
    tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)

    src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
    tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)

    encoder_blocks = []
    for _ in range(N):
        encoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
        encoder_block = EncoderBlock(d_model, encoder_self_attention_block, feed_forward_block, dropout)
        encoder_blocks.append(encoder_block)

    decoder_blocks = []
    for _ in range(N):
        decoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
        decoder_cross_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
        decoder_block = DecoderBlock(d_model, decoder_cross_attention_block, decoder_cross_attention_block, feed_forward_block, dropout)
        decoder_blocks.append(decoder_block) 


    encoder = Encoder(d_model, nn.ModuleList(encoder_blocks))
    decoder = Decoder(d_model, nn.ModuleList(decoder_blocks))

    projection_layer = ProjectionLayer(d_model, tgt_vocab_size)

    transformer = Transformer(encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, projection_layer)

    for p in transformer.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)


    return transformer

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
#import torch.utils.data as data
#import math
#import copy
#from model import  build_transformer

src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
src_seq_length = 100
dropout = 0.1
#tgt_seq_length = 100
tgt_seq_length = 100

#src_mask = None
#tgt_mask = None


transformer = build_transformer(src_vocab_size, tgt_vocab_size, src_seq_length, tgt_seq_length, d_model, num_heads, num_layers, dropout, d_ff)

def generate_mask(src, tgt):  
    src_mask = (src != 0).unsqueeze(1).unsqueeze(2) # padding mask
    tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
    seq_length = tgt.size(1)
    nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool()
    tgt_mask = tgt_mask & nopeak_mask # lookahead mask
    return src_mask, tgt_mask

# Generate random sample data
src_data = torch.randint(1, src_vocab_size, (64, src_seq_length))  # (batch_size, seq_length)
tgt_data = torch.randint(1, tgt_vocab_size, (64, tgt_seq_length))  # (batch_size, seq_length)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)


transformer.train()

for epoch in range(5):
    optimizer.zero_grad()
    src_mask, tgt_mask = generate_mask(src_data, tgt_data[:, :-1])
    #print(f"src_mask.shape: {src_mask.shape}") #torch.Size([64, 1, 1, 100]) (batch_size, 1, 1, src_seq)
    #print(f"tgt_mask.shape: {tgt_mask.shape}") #torch.Size([64, 1, 99, 99]) (batch_size, 1, tgt_seq_len, tgt_seq_len)
    #print(f"src_data.shape: {src_data.shape}")
    encoder_output = transformer.encode(src_data, src_mask)
    #print(f"encoder_output.shape: {encoder_output.shape}") 
    #print(f"tgt_data.shape: {tgt_data.shape}")
    decoder_output = transformer.decode(encoder_output, src_mask, tgt_data[:, :-1], tgt_mask)
    #print(f"decoder_output.shape: {decoder_output.shape}")
    proj_output = transformer.project(decoder_output)
    #print(f"transformer output: {proj_output.shape}")
    #print(f"tgt.shap: {tgt_data.shape}")
    loss = criterion(proj_output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    #output = transformer(src_data, tgt_data[:, :-1])
    #print(f"transformer output: {output.shape}")
    #loss = criterion(output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    loss.backward()
    optimizer.step()
    print(f"Epoch: {epoch+1}, Loss: {loss.item()}")
    
    
transformer.eval()

# Generate random sample validation data
val_src_data = torch.randint(1, src_vocab_size, (64, src_seq_length))  # (batch_size, seq_length)
val_tgt_data = torch.randint(1, tgt_vocab_size, (64, tgt_seq_length))  # (batch_size, seq_length)

with torch.no_grad():

    #val_output = transformer(val_src_data, val_tgt_data[:, :-1])
    #val_loss = criterion(val_output.contiguous().view(-1, tgt_vocab_size), val_tgt_data[:, 1:].contiguous().view(-1))
    src_mask, tgt_mask = generate_mask(val_src_data, val_tgt_data[:, :-1])
    encoder_output = transformer.encode(val_src_data, src_mask)
    decoder_output = transformer.decode(encoder_output, src_mask, val_tgt_data[:, :-1], tgt_mask)
    proj_output = transformer.project(decoder_output)
    val_loss = criterion(proj_output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
   
    #loss = criterion(proj_output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    print(f"Validation Loss: {val_loss.item()}")

Epoch: 1, Loss: 8.517269134521484
Epoch: 2, Loss: 8.517183303833008
Epoch: 3, Loss: 8.51708698272705
Epoch: 4, Loss: 8.516979217529297
Epoch: 5, Loss: 8.516860961914062
Validation Loss: 8.51672649383545


### Building a Transformer with PyTorch

https://www.datacamp.com/tutorial/building-a-transformer-with-py-torch

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value
        
        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        #print(attn_scores.shape) # torch.Size([64, 8, 100, 100]) 
        # encoder attn_scores: torch.Size([64, 8, 100, 100]) (batch_size, h, src_seq_len, src_seq_len)
        # decoder self_attn: torch.Size([64, 8, 99, 99]) (batch_size, h, tgt_seq_len, tgt_seq_len)
        # decoder cross_attention: torch.Size([64, 8, 99, 100]) (batch_size, h, tgt_seq_len, src_seq_len)
        
        #print(mask.shape) # torch.Size([64, 1, 1, 100])
        #encoder mask.shape: torch.Size([64, 1, 1, 100]) (batch_size, 1, 1, src_seq_len)
        #decoder self_attn_mask.shape: torch.Size([64, 1, 99, 99]) (batch_size, 1, tgt_seq_len, tgt_seq_len)
        #decoder cross_attn_mask.shape: torch.Size([64, 1, 1, 100]) (batch_size, 1, 1, src_seq_len)
        
        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        #print(f"attn_probs: {attn_probs.shape}")
        #print(f"V: {V.shape}")
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        #print(Q.shape) # torch.Size([64, 99, 512]) ==> decoder cross_attention
        #print(K.shape) # torch.Size([64, 100, 512])
        #print(V.shape) # torch.Size([64, 100, 512])
        
        Q = self.split_heads(self.W_q(Q)) # 
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        #print(Q.shape) # torch.Size([64, 8, 99, 64]) ==> decoder cross_attention
        #print(K.shape) # torch.Size([64, 8, 100, 64])
        #print(V.shape) # torch.Size([64, 8, 100, 64])
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        #print(f"attn_output: {attn_output.shape}")

        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [3]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [5]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [23]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        #print(x.shape) # torch.Size([64, 99, 512]) 
        #print(enc_output.shape) # torch.Size([64, 100, 512])
        #print(src_mask.shape) # torch.Size([64, 1, 1, 100])
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        #print(f"cross_attn output: {attn_output.shape}")
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

In [28]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    @staticmethod  ## added
    def generate_mask(src, tgt):  ## added
    #def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    #@staticmethod
    #def forward(src, tgt):
    def forward(self, src, tgt):
        #src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_mask, tgt_mask = Transformer.generate_mask(src, tgt)  ## added
        print(f"src_mask.shape: {src_mask.shape}")
        print(f"tgt_mask.shape: {tgt_mask.shape}")
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        #print(enc_output.shape) # torch.Size([64, 100, 512])
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)
        print(f"enc_layer output: {enc_output.shape}")

        dec_output = tgt_embedded
        #print(f"encdec_output.shape) # torch.Size([64, 99, 512])
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)
        print(f"dec_layer output: {dec_output.shape}")

        output = self.fc(dec_output)
        return output

In [29]:
src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1
dec_max_seq_length = 50

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)

# Generate random sample data
src_data = torch.randint(1, src_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)
#tgt_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)
tgt_data = torch.randint(1, tgt_vocab_size, (64, dec_max_seq_length))  # (batch_size, seq_length)

In [30]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer.train()

for epoch in range(5):
    optimizer.zero_grad()
    output = transformer(src_data, tgt_data[:, :-1])
    print(f"transformer output: {output.shape}")
    loss = criterion(output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    loss.backward()
    optimizer.step()
    print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

src_mask.shape: torch.Size([64, 1, 1, 100])
tgt_mask.shape: torch.Size([64, 1, 49, 49])
enc_layer output: torch.Size([64, 100, 512])
dec_layer output: torch.Size([64, 49, 512])
transformer output: torch.Size([64, 49, 5000])
Epoch: 1, Loss: 8.695049285888672
src_mask.shape: torch.Size([64, 1, 1, 100])
tgt_mask.shape: torch.Size([64, 1, 49, 49])
enc_layer output: torch.Size([64, 100, 512])
dec_layer output: torch.Size([64, 49, 512])
transformer output: torch.Size([64, 49, 5000])
Epoch: 2, Loss: 8.50821590423584
src_mask.shape: torch.Size([64, 1, 1, 100])
tgt_mask.shape: torch.Size([64, 1, 49, 49])
enc_layer output: torch.Size([64, 100, 512])
dec_layer output: torch.Size([64, 49, 512])
transformer output: torch.Size([64, 49, 5000])
Epoch: 3, Loss: 8.408475875854492
src_mask.shape: torch.Size([64, 1, 1, 100])
tgt_mask.shape: torch.Size([64, 1, 49, 49])
enc_layer output: torch.Size([64, 100, 512])
dec_layer output: torch.Size([64, 49, 512])
transformer output: torch.Size([64, 49, 5000])
Epo

In [27]:
transformer.eval()

# Generate random sample validation data
val_src_data = torch.randint(1, src_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)
val_tgt_data = torch.randint(1, tgt_vocab_size, (64, dec_max_seq_length))  # (batch_size, seq_length)

with torch.no_grad():

    val_output = transformer(val_src_data, val_tgt_data[:, :-1])
    val_loss = criterion(val_output.contiguous().view(-1, tgt_vocab_size), val_tgt_data[:, 1:].contiguous().view(-1))
    print(f"Validation Loss: {val_loss.item()}")

enc_layer output: torch.Size([64, 100, 512])
dec_layer output: torch.Size([64, 49, 512])
Validation Loss: 8.692550659179688


In [31]:
src = src_data  # torch.Size([64,100])
tgt = tgt_data[:, :-1] # torch.Size([64,49])

In [12]:
src_mask, tgt_mask = Transformer.generate_mask(src, tgt)

In [13]:
src_mask.shape

torch.Size([64, 1, 1, 100])

In [14]:
tgt.shape

torch.Size([64, 49])

In [15]:
(src != 0).unsqueeze(1).unsqueeze(2).shape

torch.Size([64, 1, 1, 100])

In [16]:
(tgt != 0).unsqueeze(1).shape

torch.Size([64, 1, 49])

In [17]:
(tgt != 0).unsqueeze(1).unsqueeze(3).shape

torch.Size([64, 1, 49, 1])

In [18]:
seq_length = tgt.size(1) # 49

In [34]:
nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool() # torch.Size([1, 99, 99])
print(nopeak_mask.shape)
print(nopeak_mask)

torch.Size([1, 49, 49])
tensor([[[ True, False, False,  ..., False, False, False],
         [ True,  True, False,  ..., False, False, False],
         [ True,  True,  True,  ..., False, False, False],
         ...,
         [ True,  True,  True,  ...,  True, False, False],
         [ True,  True,  True,  ...,  True,  True, False],
         [ True,  True,  True,  ...,  True,  True,  True]]])


In [33]:
tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3) # torch.Size([64, 1, 49, 1])
tgt_mask.shape

torch.Size([64, 1, 49, 1])

In [21]:
(tgt_mask & nopeak_mask).shape

torch.Size([64, 1, 49, 49])

### 